<a href="https://colab.research.google.com/github/Imtiyaj56/LangChain_Practise/blob/main/15_Tools_In_Langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


## Built-in Tool - DuckDuckGo Search

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke('top news in india today')

print(results)

Read full articles, watch videos, browse thousands of titles and more on the "India" topic with GoogleNews. Get all the latestnews, live updates and content aboutIndiafrom across the BBC. Check out the latestnewsfromIndiaand around the world. LatestIndianewson Bollywood, Politics, Business, Cricket, Technology and Travel.Stay withIndiaToday.in for the latest and live updates. 'Mossad agents in Iran': DoIndianGMs agree with World No. 2 Nakamura's FIDE jab? Chess is grappling with a cheating crisis, forcing stringent security attoptournaments. World No. 2 Hikaru Nakamura sarcastically compared players to spies due to intense scanning. Get breakingnewsalerts fromIndiaand followtoday’s livenewsupdates in field of politics, business, technology, Bollywood, cricket and more.


## Built-in Tool - Shell Tool

In [ ]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

results = shell_tool.invoke('whoami')

print(results)

Executing command:
 whoami
root



/usr/local/lib/python3.12/dist-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


## Custom Tools (Method 1)

In [ ]:
from langchain_core.tools import tool

In [ ]:
# Step 1 - Create a function for tool
def multiply(a,b):
  '''Multiply Two Numbers'''
  return a*b

In [ ]:
# Step 2 - Add a typehint
def multiply(a: int, b: int) -> int:
  '''Multiply Two  Numbers'''
  return a*b

In [ ]:
# Step 3 - Add tool decorator

@tool
def multiply(a: int, b: int) -> int:
  '''Multiply Two  Numbers'''
  return a*b

In [ ]:
result = multiply.invoke({"a":3, "b":5})
result

15

In [ ]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply Two  Numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## Method 2 - Using StructuredTool

In [ ]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [ ]:
class MultiplyInput(BaseModel):
  a: int = Field(required=True, description='First no to multply')
  b: int = Field(required=True, description='second no to multply')

/tmp/ipykernel_3582/4113385663.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  a: int = Field(required=True, description='First no to multply')
/tmp/ipykernel_3582/4113385663.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  b: int = Field(required=True, description='second no to multply')


In [ ]:
def multiply_func(a: int, b: int) -> int:
    return a * b

In [ ]:
multiply_tool = StructuredTool.from_function(
    func = multiply_func,
    name = 'multiply2',
    description= 'Multiply two numbers',
    args_schema=MultiplyInput
)

In [ ]:
result = multiply_tool.invoke({'a':3, 'b':6})
result

18

In [ ]:
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

multiply2
Multiply two numbers
{'a': {'description': 'First no to multply', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'second no to multply', 'title': 'B', 'type': 'integer'}}


## Method 3 - Using BaseTool Class

In [ ]:
from langchain_core.tools import BaseTool
from typing import Type

In [ ]:
# schema using pydantic

class MultiplyInput(BaseModel):
  a: int = Field(required=True, description='First no to multply')
  b: int = Field(required=True, description='second no to multply')

/tmp/ipykernel_3582/1984935078.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  a: int = Field(required=True, description='First no to multply')
/tmp/ipykernel_3582/1984935078.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  b: int = Field(required=True, description='second no to multply')


In [ ]:
class MultiplyTool(BaseTool):
    name: str = "multiply3"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

In [ ]:
multiply_tool3 = MultiplyTool()

In [ ]:
result = multiply_tool3.invoke({'a': 15, 'b': 10})
result

150

## ToolKit

In [ ]:
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


In [ ]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]

In [ ]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)

add => Add two numbers
multiply => Multiply two numbers
